In [0]:
%sql
SELECT *
FROM healthcare_dev.bronze.patients_raw;

In [0]:
# ============================================================
# SILVER LAYER — DIM_PATIENT
# Source: healthcare_dev.bronze.patients_raw
# Target: healthcare_dev.silver.dim_patient
# ============================================================

from pyspark.sql.functions import (
    sha2,
    col,
    current_timestamp,
    concat_ws
)
from delta.tables import DeltaTable

# 1. Configure Bronze source, Silver target, and checkpoint

bronze_table = "healthcare_dev.bronze.patients_raw"
silver_table = "healthcare_dev.silver.dim_patient"

checkpoint_path = (
    "abfss://healthcare@ecommerceunifiedproject.dfs.core.windows.net/"
    "silver/dim_patient/checkpoint/"
)

# 2. Read Bronze using Structured Streaming

df_patient_bronze = spark.readStream.table(bronze_table)

# 3. Deduplicate, hash patient names, and add timestamp

df_patient_clean = (
    df_patient_bronze
    .dropDuplicates(["patient_id"])
    .withColumn(
        "patient_first_last_name_masked",
        sha2(
            concat_ws("|", col("first_name"), col("last_name")),
            256
        )
    )
    .withColumn("load_timestamp", current_timestamp())
)

# 4. Define Delta MERGE operation

def merge_dim_patient(batch_df, batch_id):

    if not spark.catalog.tableExists(silver_table):
        (
            batch_df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(silver_table)
        )
        return

    dim_patient = DeltaTable.forName(spark, silver_table)

    (
        dim_patient.alias("t")
        .merge(
            batch_df.alias("s"),
            "t.patient_id = s.patient_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

# 5. Execute incremental streaming ingestion

query = (
    df_patient_clean.writeStream
    .foreachBatch(merge_dim_patient)
    .outputMode("update")
    .trigger(availableNow=True)
    .option("checkpointLocation", checkpoint_path)
    .start()
)

query.awaitTermination()

print("Silver dim_patient ingestion completed.")

# 6. Verify Silver table

display(
    spark.table("healthcare_dev.silver.dim_patient")
)